# Create Agents to Research and Write an Article with CrewAi

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29 crewai[litellm] crewai[anthropic]
```

In [1]:
# Warning control:
import warnings
import os

warnings.filterwarnings('ignore')

# Disable OpenTelemetry SDK to prevent automatic tracing initialization
os.environ["OTEL_SDK_DISABLED"] = "true"

- Import from the crewAI libray.

In [4]:
from crewai import Agent, Task, Crew

12:57:29 - LiteLLM:WARNING: get_model_cost_map.py:271 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: timed out. Falling back to local backup.


The ```.env``` file is configured with:

```MODEL_NAME="openai/gemini-3-pro-preview"```
    
```OPENAI_BASE_URL="https://api.gapgpt.app/v1"```

In [6]:
from dotenv import load_dotenv
import os

load_dotenv()

True

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [10]:
planner = Agent(
    role="برنامه‌ریز محتوا",
    goal="برنامه‌ریزی محتوای جذاب و دقیق درباره موضوع {topic}",
    backstory="شما در حال برنامه‌ریزی برای یک مقاله وبلاگ "
              "درباره موضوع: {topic} هستید. "
              "شما اطلاعاتی جمع‌آوری می‌کنید که به مخاطبان کمک می‌کند "
              "چیز جدیدی یاد بگیرند "
              "و تصمیمات آگاهانه‌ای بگیرند. "
              "کار شما پایه و اساس کار نویسنده محتوا "
              "برای نوشتن مقاله درباره این موضوع است.",
    allow_delegation=False,
    verbose=True
)

### Agent: Writer

In [13]:
writer = Agent(
    role="نویسنده محتوا",
    goal="نوشتن یک مقاله نظری جذاب و دقیق "
         "درباره موضوع: {topic}",
    backstory="شما در حال نوشتن یک مقاله نظری جدید "
              "درباره موضوع: {topic} هستید. "
              "نوشته شما بر اساس کار برنامه‌ریز محتوا است، "
              "که یک طرح کلی و زمینه مرتبط "
              "درباره موضوع ارائه می‌دهد. "
              "شما از اهداف اصلی و مسیر طرح کلی "
              "که توسط برنامه‌ریز محتوا ارائه شده پیروی می‌کنید. "
              "همچنین دیدگاه‌های عینی و بی‌طرفانه ارائه می‌دهید "
              "و آن‌ها را با اطلاعات ارائه شده "
              "توسط برنامه‌ریز محتوا پشتیبانی می‌کنید. "
              "در مقاله نظری خود مشخص می‌کنید "
              "که کدام جملات نظر شخصی هستند "
              "و کدام‌ها بیانات عینی و واقعی می‌باشند.",
    allow_delegation=False,
    verbose=True
)


### Agent: Editor

In [16]:
editor = Agent(
    role="ویراستار",
    goal="ویرایش مقاله وبلاگ دریافتی "
         "و هماهنگ کردن آن با سبک نوشتاری سازمان.",
    backstory="شما یک ویراستار هستید که مقاله وبلاگ را "
              "از نویسنده محتوا دریافت می‌کنید. "
              "هدف شما بررسی مقاله وبلاگ است "
              "تا مطمئن شوید که از بهترین شیوه‌های روزنامه‌نگاری پیروی می‌کند، "
              "هنگام ارائه نظرات یا ادعاها "
              "دیدگاه‌های متعادل و منصفانه ارائه می‌دهد، "
              "و تا حد امکان از موضوعات "
              "یا نظرات بحث‌برانگیز اجتناب می‌کند.",
    allow_delegation=False,
    verbose=True
)


## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [20]:
plan = Task(
    description=(
        "1. آخرین روندها، بازیگران کلیدی "
            "و اخبار مهم درباره {topic} را اولویت‌بندی کنید.\n"
        "2. مخاطبان هدف را شناسایی کنید، "
            "با توجه به علایق و نقاط درد آن‌ها.\n"
        "3. یک طرح کلی محتوای دقیق شامل "
            "مقدمه، نکات کلیدی و دعوت به اقدام تهیه کنید.\n"
        "4. کلمات کلیدی SEO و داده‌ها یا منابع مرتبط را درج کنید."
    ),
    expected_output="یک سند برنامه محتوای جامع "
        "شامل طرح کلی، تحلیل مخاطبان، "
        "کلمات کلیدی SEO و منابع.",
    agent=planner,
)


### Task: Write

In [23]:
write = Task(
    description=(
        "1. از برنامه محتوا برای نوشتن یک مقاله وبلاگ "
            "جذاب درباره {topic} استفاده کنید.\n"
        "2. کلمات کلیدی SEO را به صورت طبیعی در متن بگنجانید.\n"
        "3. بخش‌ها و زیرعنوان‌ها به شکلی جذاب "
            "و مناسب نام‌گذاری شوند.\n"
        "4. مطمئن شوید مقاله دارای ساختار مناسب است: "
            "مقدمه‌ای جذاب، متن اصلی پربار "
            "و نتیجه‌گیری خلاصه‌وار.\n"
        "5. متن را از نظر خطاهای دستوری "
            "و هماهنگی با لحن برند بازبینی کنید.\n"
    ),
    expected_output="یک مقاله وبلاگ خوب نوشته شده "
        "در قالب Markdown، آماده برای انتشار، "
        "هر بخش باید ۲ یا ۳ پاراگراف داشته باشد.",
    agent=writer,
)


### Task: Edit

In [26]:
edit = Task(
    description=("مقاله وبلاگ داده شده را از نظر "
                 "خطاهای دستوری و "
                 "هماهنگی با لحن برند بازبینی و ویرایش کنید."),
    expected_output="یک مقاله وبلاگ خوب نوشته شده در قالب Markdown، "
                    "آماده برای انتشار، "
                    "هر بخش باید ۲ یا ۳ پاراگراف داشته باشد.",
    agent=editor
)


## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution. 

In [31]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=True
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [35]:
result = crew.kickoff(inputs={"topic": "هوش مصنوعی"})

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.1                                                                                        │
│  Latest version:  1.14.2                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f56d31cc-60fb-4639-bd7b-1f2257c69520                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. آخرین روندها، بازیگران کلیدی و اخبار مهم درباره هوش مصنوعی را اولویت‌بندی کنید.                        │
│  2. مخاطبان هدف را شناسایی کنید، با توجه به علایق و نقاط درد آن‌ها.                                              │
│  3. یک طرح کلی محتوای دقیق شامل مقدمه، نکات کلیدی و دعوت به اقدام تهیه کنید.                                    │
│  4. کلمات کلیدی SEO و داده‌ها یا منابع مرتبط را درج کنید.                                                        │
│  ID: b872f6c3-818f-4d17-b17a-db4f61c5fd23                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: برنامه‌ریز محتوا                                                                                         │
│                                                                                                                 │
│  Task: 1. آخرین روندها، بازیگران کلیدی و اخبار مهم درباره هوش مصنوعی را اولویت‌بندی کنید.                        │
│  2. مخاطبان هدف را شناسایی کنید، با توجه به علایق و نقاط درد آن‌ها.                                              │
│  3. یک طرح کلی محتوای دقیق شامل مقدمه، نکات کلیدی و دعوت به اقدام تهیه کنید.                                    │
│  4. کلمات کلیدی SEO و داده‌ها یا منابع مرتبط را درج کنید.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: برنامه‌ریز محتوا                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **سند جامع برنامه‌ریزی محتوا: راهنمای کاربردی هوش مصنوعی (AI) برای سال ۲۰۲۴**                                   │
│                                                                                                                 │
│  این سند به عنوان نقشه راه و پایه و اساس کار نویسنده محتوا تهیه شده است تا مقاله‌ای جذاب، دقیق، سئو شده و        │
│  ارزش‌آفرین خلق کند که به مخاطب در یادگیری و تصمیم‌گیری آگاهانه کمک نماید.                                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### فاز ۱: اولویت‌بندی روندها، بازیگران کلیدی و اخبار مهم هوش مصنوعی                                            │
│                                                                                                                 │
│  برای اینکه مقاله کاملاً به‌روز و معتبر باشد، نویسنده باید روی این محورها تمرکز کند:                              │
│                                                                                                                 │
│  *   **آخرین روندهای کلیدی (Trends):**                                                                          │
│      *   **هوش مصنوعی مولد (Generative AI):** فراتر رفتن از تولید متن و ورود قدرتمند به تولید ویدیو (مانند      │
│  Sora)، موسیقی و کدنویسی.                                                                                       │
│      *   **دستیاران هوشمند یکپارچه (Copilots):** ادغام هوش مصنوعی در نرم‌افزارهای روزمره (مانند Microsoft        │
│  Copilot در آفیس و Google Workspace).                                                                           │
│      *   **شخصی‌سازی (Personalization):** توانایی ساخت مدل‌های هوش مصنوعی شخصی‌سازی شده (GPTs) برای کارهای خاص     │
│  بدون نیاز به دانش کدنویسی.                                                                                     │
│      *   **قوانین و اخلاق در هوش مصنوعی (AI Ethics & Regulation):** تمرکز جهانی بر کپی‌رایت، حفظ حریم خصوصی و    │
│  امنیت داده‌ها.                                                                                                  │
│  *   **بازیگران کلیدی (Key Players):**                                                                          │
│      *   **OpenAI:** پیشگام با مدل‌های GPT-4o و Sora.                                                            │
│      *   **Google:** با مدل‌های زبانی Gemini و ادغام آن در موتور جستجو.                                          │
│      *   **Microsoft:** بزرگترین سرمایه‌گذار OpenAI و توسعه‌دهنده Copilot.                                        │
│      *   **Anthropic:** با مدل Claude 3 (تمرکز بر ایمنی و درک متن بالا).                                        │
│      *   **Meta:** با مدل متن‌باز Llama 3.                                                                       │
│  *   **اخبار مهم و تاثیرگذار:**                                                                                 │
│      *   تصویب «قانون هوش مصنوعی اروپا» (EU AI Act) به عنوان اولین چارچوب قانونی جامع در جهان.                  │
│      *   بحث‌های داغ پیرامون جایگزینی مشاغل توسط AI در مقابل ایجاد فرصت‌های شغلی جدید (تکامل مشاغل).              │
│                                   

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: ویراستار                                                                                                │
│                                                                                                                 │
│  Task: مقاله وبلاگ داده شده را از نظر خطاهای دستوری و هماهنگی با لحن برند بازبینی و ویرایش کنید.                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: ویراستار                                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  در ادامه، نسخه ویرایش‌شده و نهایی مقاله وبلاگ شما ارائه شده است. در این ویرایش، تمام توضیحات درون‌پرانتزی        │
│  نویسنده و اظهارنظرهای شخصی (مانند "به نظر من") حذف شده‌اند تا متن لحنی حرفه‌ای، روزنامه‌نگارانه و کاملاً منطبق با  │
│  صدای برند (امیدوارکننده و آموزشی) پیدا کند. همچنین ساختار مقاله به گونه‌ای تنظیم شده است که **هر بخش دقیقاً      │
│  دارای ۲ یا ۳ پاراگراف** باشد.                                                                                  │
│                                                                                                                 │
│  ***                                                                                                            │
│                                                                                                                 │
│  # راهنمای بقا در عصر هوش مصنوعی: چگونه در سال ۲۰۲۴ از رقبا عقب نمانیم؟                                         │
│                                                                                                                 │
│  شاید تا همین چند سال پیش، وقتی عبارت **هوش مصنوعی چیست** را می‌شنیدیم، یاد ربات‌های فیلم‌های علمی‌تخیلی            │
│  می‌افتادیم. اما امروز، هوش مصنوعی دیگر یک افسانه نیست؛ بلکه تکنولوژی قدرتمندی است که همین الان در گوشی موبایل،  │
│  لپ‌تاپ و حتی نرم‌افزارهای روزمره ما در حال کار است. بر اساس گزارش‌های معتبر جهانی، پیش‌بینی می‌شود حجم بازار هوش    │
│  مصنوعی تا سال ۲۰۳۰ به بیش از ۷۳۸ میلیارد دلار برسد که این عدد نشان‌دهنده یک روند صعودی و غیرقابل توقف در دنیای  │
│  تکنولوژی است.                                                                                                  │
│                                                                                                                 │
│  با این سرعت رشد خیره‌کننده، عدم آشنایی با **کاربرد هوش مصنوعی** می‌تواند به قیمت جا ماندن از بازار کار رقابتی    │
│  امروز تمام شود. طبق بررسی‌های موسسه McKinsey، هوش مصنوعی مولد می‌تواند سالانه بین ۲.۶ تا ۴.۴ تریلیون دلار به     │
│  اقتصاد جهانی اضافه کند. این اعداد بزرگ نباید باعث سردرگمی شوند؛ بلکه باید به عنوان یک فرصت طلایی برای رشد      │
│  دیده شوند. در این مقاله به زبان ساده بررسی می‌کنیم که هوش مصنوعی در حال حاضر چه جایگاهی دارد، چه تاثیری روی     │
│  شغل شما می‌گذارد و چطور می‌توانید از همین امروز، استفاده از آن را شروع کنید.                                     │
│                                                                                                                 │
│  ## نگاهی به بازیگران اصلی و آینده هوش مصنوعی در سال جاری                                                       │
│                                                                                                                 │
│  وقتی از ترندهای روز صحبت می‌کنیم، «هوش مصنوعی مولد» (Generative AI) حرف اول را می‌زند. این تکنولوژی که تا پیش    │
│  از این بیشتر بر تولید متن متمرکز بود، امروزه با ابزارهایی مانند Sora به شکلی شگفت‌انگیز وارد حوزه تولید ویدیو،  │
│  ساخت موسیقی و حتی برنامه‌نویسی شده است. هم‌زمان با این پیشرفت‌ها، تصویب «قانون هوش مصنوعی اروپا» (EU AI Act) به   │
│  عنوان اولین چارچوب قانونی جامع در جهان، نشان داد که دولت‌ها نیز برای قانون‌گذاری و حفظ امنیت در این فضا قدم‌های   │
│  جدی برداشته‌اند.                                                                                                │
│                                                                                                                 │
│  در این میان، غول‌ه

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: مقاله وبلاگ داده شده را از نظر خطاهای دستوری و هماهنگی با لحن برند بازبینی و ویرایش کنید.                │
│  Agent: ویراستار                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f56d31cc-60fb-4639-bd7b-1f2257c69520                                                                       │
│  Final Output: در ادامه، نسخه ویرایش‌شده و نهایی مقاله وبلاگ شما ارائه شده است. در این ویرایش، تمام توضیحات      │
│  درون‌پرانتزی نویسنده و اظهارنظرهای شخصی (مانند "به نظر من") حذف شده‌اند تا متن لحنی حرفه‌ای، روزنامه‌نگارانه و     │
│  کاملاً منطبق با صدای برند (امیدوارکننده و آموزشی) پیدا کند. همچنین ساختار مقاله به گونه‌ای تنظیم شده است که      │
│  **هر بخش دقیقاً دارای ۲ یا ۳ پاراگراف** باشد.                                                                   │
│                                                                                                                 │
│  ***                                                                                                            │
│                                                                                                                 │
│  # راهنمای بقا در عصر هوش مصنوعی: چگونه در سال ۲۰۲۴ از رقبا عقب نمانیم؟                                         │
│                                                                                                                 │
│  شاید تا همین چند سال پیش، وقتی عبارت **هوش مصنوعی چیست** را می‌شنیدیم، یاد ربات‌های فیلم‌های علمی‌تخیلی            │
│  می‌افتادیم. اما امروز، هوش مصنوعی دیگر یک افسانه نیست؛ بلکه تکنولوژی قدرتمندی است که همین الان در گوشی موبایل،  │
│  لپ‌تاپ و حتی نرم‌افزارهای روزمره ما در حال کار است. بر اساس گزارش‌های معتبر جهانی، پیش‌بینی می‌شود حجم بازار هوش    │
│  مصنوعی تا سال ۲۰۳۰ به بیش از ۷۳۸ میلیارد دلار برسد که این عدد نشان‌دهنده یک روند صعودی و غیرقابل توقف در دنیای  │
│  تکنولوژی است.                                                                                                  │
│                                                                                                                 │
│  با این سرعت رشد خیره‌کننده، عدم آشنایی با **کاربرد هوش مصنوعی** می‌تواند به قیمت جا ماندن از بازار کار رقابتی    │
│  امروز تمام شود. طبق بررسی‌های موسسه McKinsey، هوش مصنوعی مولد می‌تواند سالانه بین ۲.۶ تا ۴.۴ تریلیون دلار به     │
│  اقتصاد جهانی اضافه کند. این اعداد بزرگ نباید باعث سردرگمی شوند؛ بلکه باید به عنوان یک فرصت طلایی برای رشد      │
│  دیده شوند. در این مقاله به زبان ساده بررسی می‌کنیم که هوش مصنوعی در حال حاضر چه جایگاهی دارد، چه تاثیری روی     │
│  شغل شما می‌گذارد و چطور می‌توانید از همین امروز، استفاده از آن را شروع کنید.                                     │
│                                                                                                                 │
│  ## نگاهی به بازیگران اصلی و آینده هوش مصنوعی در سال جاری                                                       │
│                                                                                                                 │
│  وقتی از ترندهای روز صحبت می‌کنیم، «هوش مصنوعی مولد» (Generative AI) حرف اول را می‌زند. این تکنولوژی که تا پیش    │
│  از این بیشتر بر تولید متن متمرکز بود، امروزه با ابزارهایی مانند Sora به شکلی شگفت‌انگیز وارد حوزه تولید ویدیو،  │
│  ساخت موسیقی و حتی برنامه‌نویسی شده است. هم‌زمان با این پیشرفت‌ها، تصویب «قانون هوش مصنوعی اروپا» (EU AI Act) به   │
│  عنوان اولین چارچوب قانونی جامع در جهان، نشان داد که دولت‌ها نیز برای قانون‌گذاری و حفظ امنیت در این فضا قدم‌های   │
│  جدی برداشته‌اند.                                                                                                │
│                                                                                                                 │
│  در این میان، غول‌

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

- Display the results of your execution as markdown in the notebook.

In [36]:
from IPython.display import HTML
import markdown

html_content = markdown.markdown(result.raw)

HTML(f'''
<style>
    .rtl-content * {{
        direction: rtl !important;
        text-align: right !important;
    }}
</style>
<div class="rtl-content" style="
    direction: rtl;
    text-align: right; 
    font-family: Tahoma, Arial, sans-serif; 
    font-size: 15px; 
    line-height: 2;
    padding: 20px;
">
    {html_content}
</div>
''')


## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [ ]:
topic = "YOUR TOPIC HERE"
result = crew.kickoff(inputs={"topic": topic})

In [ ]:
Markdown(result)

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).